In [ ]:

import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No GPU detected. Select a GPU runtime.")


Python: 3.13.16
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:

import torch
import transformers
import bitsandbytes as bnb
import peft
import trl
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bnb.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

print("\nGPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

# Test GPU tensor operation
x = torch.randn(100, 100, device="cuda")
print("GPU test:", (x @ x).shape)


PyTorch: 2.14.1+cu130
Transformers: 5.22.1
BitsAndBytes: 0.50.2
PEFT: 0.21.2
TRL: 1.17.1
Datasets: 4.8.5
Accelerate: 1.15.0

GPU: Tesla T4
CUDA: True
GPU test: torch.Size([100, 100])


In [ ]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Existing baseline project directory
PROJECT_DIR = Path("/content/drive/MyDrive/StructuraAI")

# Fine-tuning output directory
OUTPUT_DIR = PROJECT_DIR / "finetuned_aya_expanse_8b"

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
ADAPTER_DIR = OUTPUT_DIR / "adapter"
LOG_DIR = OUTPUT_DIR / "training_logs"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
EVALUATION_DIR = OUTPUT_DIR / "evaluation"

for folder in [
    CHECKPOINT_DIR,
    ADAPTER_DIR,
    LOG_DIR,
    PREDICTION_DIR,
    EVALUATION_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Output directories ready!")
print("Saving results to:", OUTPUT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output directories ready!
Saving results to: /content/drive/MyDrive/StructuraAI/finetuned_aya_expanse_8b


In [ ]:

from datasets import load_dataset

DATASET_ID = "rakesh-bishnoi29/structuraai-dataset"

dataset = load_dataset(DATASET_ID)

print(dataset)
print("\nAvailable splits:", list(dataset.keys()))


Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['document_id', 'image', 'input', 'output', 'annotation_file', 'synthetic'],
        num_rows: 196
    })
    validation: Dataset({
        features: ['document_id', 'image', 'input', 'output', 'annotation_file', 'synthetic'],
        num_rows: 42
    })
    test: Dataset({
        features: ['document_id', 'image', 'input', 'output', 'annotation_file', 'synthetic'],
        num_rows: 42
    })
})

Available splits: ['train', 'validation', 'test']


In [ ]:
import getpass
from huggingface_hub import login, whoami

# Enter your Hugging Face access token securely
HF_TOKEN = getpass.getpass("Paste your Hugging Face Access Token: ")

# Authenticate the account
login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

# Verify authenticated account
account = whoami(token=HF_TOKEN)

print("\nHugging Face authentication successful!")
print("Authenticated account:", account["name"])

Paste your Hugging Face Access Token: ··········

Hugging Face authentication successful!
Authenticated account: rakesh-bishnoi29


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_ID = "CohereLabs/aya-expanse-8b"

try:
    config_path = hf_hub_download(
        repo_id=MODEL_ID,
        filename="config.json"
    )

    print("SUCCESS: Aya Expanse 8B access verified!")
    print("Config downloaded:", config_path)

except Exception as e:
    print("MODEL ACCESS VERIFICATION FAILED")
    print(type(e).__name__, str(e))

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

SUCCESS: Aya Expanse 8B access verified!
Config downloaded: /root/.cache/huggingface/hub/models--CohereLabs--aya-expanse-8b/snapshots/5062468bf9bc0c6035fd64e06274333ec127d980/config.json


In [ ]:

import json
from datetime import datetime, date
from datasets import Dataset

CANONICAL_FIELDS = [
    "supplier", "customer", "invoice_number",
    "invoice_date", "due_date", "purchase_order_number",
    "currency", "line_items", "subtotal", "tax_total",
    "total", "payment_terms", "payment_reference", "notes"
]

SUPPLIER_FIELDS = [
    "name", "address", "phone", "email", "website"
]

LINE_ITEM_FIELDS = [
    "description", "quantity", "unit_price", "tax_rate",
    "line_subtotal", "line_tax", "line_total"
]

def serialize_dates(value):
    if isinstance(value, datetime):
        return value.date().isoformat()
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, dict):
        return {k: serialize_dates(v) for k, v in value.items()}
    if isinstance(value, list):
        return [serialize_dates(v) for v in value]
    return value

def clean_invoice_target(raw):
    missing = [k for k in CANONICAL_FIELDS if k not in raw]
    if missing:
        raise ValueError(f"Missing fields: {missing}")

    cleaned = {k: raw[k] for k in CANONICAL_FIELDS}

    cleaned["supplier"] = {
        k: raw["supplier"][k]
        for k in SUPPLIER_FIELDS
        if k in raw["supplier"]
    }

    cleaned["line_items"] = [
        {
            k: item[k]
            for k in LINE_ITEM_FIELDS
            if k in item
        }
        for item in raw["line_items"]
    ]

    return serialize_dates(cleaned)

def prepare_split(split_name):
    records = []

    for row in dataset[split_name]:
        records.append({
            "document_id": row["document_id"],
            "prompt": row["input"],
            "completion": json.dumps(
                clean_invoice_target(row["output"]),
                ensure_ascii=False,
                separators=(",", ":")
            )
        })

    return Dataset.from_list(records)

train_prepared = prepare_split("train")
validation_prepared = prepare_split("validation")
test_prepared = prepare_split("test")

print("Training:", len(train_prepared))
print("Validation:", len(validation_prepared))
print("Test:", len(test_prepared))
print("Preparation successful!")


Training: 196
Validation: 42
Test: 42
Preparation successful!


In [ ]:

from transformers import AutoTokenizer

MODEL_ID = "CohereLabs/aya-expanse-8b"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True
)

print("Aya tokenizer loaded successfully!")
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)


Aya tokenizer loaded successfully!
EOS token: <|END_OF_TURN_TOKEN|>
PAD token: <PAD>
Chat template available: True


In [ ]:

import numpy as np

def get_token_length(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokens = tokenizer(
        formatted_text,
        add_special_tokens=False
    )["input_ids"]

    return len(tokens)


train_lengths = [
    get_token_length(example)
    for example in train_prepared
]

validation_lengths = [
    get_token_length(example)
    for example in validation_prepared
]


def print_statistics(name, lengths):
    print(f"\n{'='*40}")
    print(name)

    print("Examples:", len(lengths))
    print("Maximum:", max(lengths))
    print("Average:", round(np.mean(lengths), 2))
    print("Median:", int(np.median(lengths)))
    print("95th percentile:", int(np.percentile(lengths, 95)))

    print("Above 4096:", sum(x > 4096 for x in lengths))
    print("Above 6144:", sum(x > 6144 for x in lengths))
    print("Above 8192:", sum(x > 8192 for x in lengths))


print_statistics("TRAIN", train_lengths)
print_statistics("VALIDATION", validation_lengths)



TRAIN
Examples: 196
Maximum: 1107
Average: 862.59
Median: 861
95th percentile: 1079
Above 4096: 0
Above 6144: 0
Above 8192: 0

VALIDATION
Examples: 42
Maximum: 1096
Average: 856.57
Median: 859
95th percentile: 1077
Above 4096: 0
Above 6144: 0
Above 8192: 0


In [ ]:

from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True
)

print("Model architecture:", config.architectures)
print("Hidden size:", config.hidden_size)
print("Number of layers:", config.num_hidden_layers)
print("Attention heads:", config.num_attention_heads)
print("Maximum position embeddings:", config.max_position_embeddings)


Model architecture: ['CohereForCausalLM']
Hidden size: 4096
Number of layers: 32
Attention heads: 32
Maximum position embeddings: 8192


In [ ]:

def create_prompt_completion(example):
    user_message = {
        "role": "user",
        "content": example["prompt"]
    }

    assistant_message = {
        "role": "assistant",
        "content": example["completion"]
    }

    # User message plus Aya's assistant-generation prefix
    prompt_text = tokenizer.apply_chat_template(
        [user_message],
        tokenize=False,
        add_generation_prompt=True
    )

    # Complete conversation including assistant end-of-turn token
    full_text = tokenizer.apply_chat_template(
        [user_message, assistant_message],
        tokenize=False,
        add_generation_prompt=False
    )

    if not full_text.startswith(prompt_text):
        raise ValueError(
            "Aya chat template prompt boundary mismatch."
        )

    completion_text = full_text[len(prompt_text):]

    return {
        "prompt": prompt_text,
        "completion": completion_text
    }


train_sft = train_prepared.map(
    create_prompt_completion,
    remove_columns=train_prepared.column_names
)

validation_sft = validation_prepared.map(
    create_prompt_completion,
    remove_columns=validation_prepared.column_names
)

print("Training SFT examples:", len(train_sft))
print("Validation SFT examples:", len(validation_sft))

print("\nPROMPT END:")
print(repr(train_sft[0]["prompt"][-150:]))

print("\nCOMPLETION START:")
print(repr(train_sft[0]["completion"][:200]))

print("\nCOMPLETION END:")
print(repr(train_sft[0]["completion"][-100:]))

assert len(train_sft) == 196
assert len(validation_sft) == 42
assert tokenizer.eos_token in train_sft[0]["completion"]

print("\nAya prompt-completion preparation successful!")


Map:   0%|          | 0/196 [00:00<?, ? examples/s]

Map:   0%|          | 0/42 [00:00<?, ? examples/s]

Training SFT examples: 196
Validation SFT examples: 42

PROMPT END:
'CCT-169789\nPlease quote the invoice number with your payment.\nSYNTHETIC TRAINING DOCUMENT<|END_OF_TURN_TOKEN|><|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|>'

COMPLETION START:
'{"supplier":{"name":"Orchard Peak Foods","address":"8 Harvest Street, Rotterdam 3011 AA, Netherlands","phone":"+91 80 4431 7464","email":"billing@orchard.example","website":"www.orchard.example"},"cus'

COMPLETION END:
'ce":"ACCT-169789","notes":"Please quote the invoice number with your payment."}<|END_OF_TURN_TOKEN|>'

Aya prompt-completion preparation successful!


In [ ]:

import torch
import gc

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import prepare_model_for_kbit_training

MODEL_ID = "CohereLabs/aya-expanse-8b"

MAX_SEQ_LENGTH = 2048

# Confirm GPU
assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nLoading Aya Expanse 8B...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

# Disable cache during gradient-checkpointed training
model.config.use_cache = False

# Prepare quantized model for adapter training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("\nAya model loaded successfully!")

print("Model architecture:", model.config.architectures)
print("Quantization:", model.is_loaded_in_4bit)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


GPU: Tesla T4

Loading Aya Expanse 8B...


model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


Aya model loaded successfully!
Model architecture: ['CohereForCausalLM']
Quantization: True
GPU memory allocated: 7.27 GB


In [ ]:

import bitsandbytes as bnb

linear_modules = set()

for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        linear_modules.add(name.split(".")[-1])

print("Available 4-bit linear module names:")

for name in sorted(linear_modules):
    print("-", name)

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


Available 4-bit linear module names:
- down_proj
- gate_proj
- k_proj
- o_proj
- q_proj
- up_proj
- v_proj

GPU memory allocated: 7.27 GB
GPU memory reserved: 9.36 GB


In [ ]:

from peft import LoraConfig, get_peft_model

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("\nLoRA adapters applied successfully!")

model.print_trainable_parameters()



LoRA adapters applied successfully!
trainable params: 41,943,040 || all params: 8,069,976,064 || trainable%: 0.5197


In [ ]:

import inspect
import trl

from trl import SFTConfig, SFTTrainer

print("TRL version:", trl.__version__)

print("\nSFTTrainer parameters:")
print(inspect.signature(SFTTrainer.__init__))

print("\nSFTConfig parameters:")
config_parameters = inspect.signature(SFTConfig.__init__).parameters

important_parameters = [
    "output_dir",
    "learning_rate",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "num_train_epochs",
    "max_length",
    "completion_only_loss",
    "fp16",
    "bf16",
    "gradient_checkpointing",
    "eval_strategy",
    "save_strategy",
    "load_best_model_at_end",
    "optim",
    "logging_steps"
]

for name in important_parameters:
    print(f"{name}: {name in config_parameters}")


TRL version: 1.17.1

SFTTrainer parameters:
(self, model: 'str | PreTrainedModel | PeftModel', args: trl.trainer.sft_config.SFTConfig | transformers.training_args.TrainingArguments | None = None, data_collator: collections.abc.Callable[[list[typing.Any]], dict[str, typing.Any]] | None = None, train_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | None = None, eval_dataset: datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset | datasets.dataset_dict.DatasetDict | datasets.dataset_dict.IterableDatasetDict | dict[str, datasets.arrow_dataset.Dataset | datasets.iterable_dataset.IterableDataset] | None = None, processing_class: transformers.tokenization_utils_base.PreTrainedTokenizerBase | transformers.processing_utils.ProcessorMixin | None = None, compute_loss_func: collections.abc.Callable | None = None, compute_metrics: collections.abc.Callable[[transformers.trainer_utils.EvalPrediction], dict] | None = None, callbacks: list[trans

In [ ]:
import inspect
from trl import SFTConfig

supported = inspect.signature(SFTConfig.__init__).parameters

print("Warmup ratio supported:", "warmup_ratio" in supported)
print("Warmup steps supported:", "warmup_steps" in supported)

Warmup ratio supported: False
Warmup steps supported: True


In [ ]:

from pathlib import Path
import inspect
from trl import SFTConfig

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/StructuraAI/finetuned_aya_expanse_8b"
)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "training_logs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Requested configuration
requested_config = {
    "output_dir": str(CHECKPOINT_DIR),

    "num_train_epochs": 3,
    "learning_rate": 5e-5,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 4,
    "weight_decay": 0.01,
    "max_grad_norm": 0.3,
    "optim": "paged_adamw_8bit",

    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,

    "max_length": 2048,
    "packing": False,
    "completion_only_loss": True,

    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {
        "use_reentrant": False
    },

    "eval_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_total_limit": 3,

    "logging_strategy": "steps",
    "logging_steps": 5,
    "logging_dir": str(LOG_DIR),
    "report_to": "none",

    "seed": 42,
    "data_seed": 42,
    "dataloader_num_workers": 0
}

# Detect exact supported parameters
supported_parameters = set(
    inspect.signature(SFTConfig.__init__).parameters
)

compatible_config = {
    key: value
    for key, value in requested_config.items()
    if key in supported_parameters
}

unsupported_parameters = [
    key for key in requested_config
    if key not in supported_parameters
]

print("Unsupported parameters skipped:", unsupported_parameters)

# Create configuration
training_args = SFTConfig(**compatible_config)

print("\nTraining configuration created successfully!")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Optimizer:", training_args.optim)
print("Maximum length:", training_args.max_length)
print("Completion-only loss:", training_args.completion_only_loss)
print("Output:", training_args.output_dir)


Unsupported parameters skipped: ['logging_dir']

Training configuration created successfully!
Epochs: 3
Learning rate: 5e-05
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT
Maximum length: 2048
Completion-only loss: True
Output: /content/drive/MyDrive/StructuraAI/finetuned_aya_expanse_8b/checkpoints


In [ ]:

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    processing_class=tokenizer
)

print("SFTTrainer initialized successfully!")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Trainer model:", type(trainer.model).__name__)


Adding EOS to train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

SFTTrainer initialized successfully!
Training examples: 196
Validation examples: 42
Trainer model: PeftModelForCausalLM


In [ ]:

sample = trainer.train_dataset[0]

test_batch = trainer.data_collator([sample])

labels = test_batch["labels"]
attention_mask = test_batch["attention_mask"]

supervised_tokens = (labels != -100).sum().item()
total_tokens = attention_mask.sum().item()

print("Total non-padding tokens:", total_tokens)
print("Tokens contributing to loss:", supervised_tokens)
print("Ignored tokens:", total_tokens - supervised_tokens)

assert supervised_tokens > 0, "No completion tokens are supervised!"

assert supervised_tokens < total_tokens, (
    "Prompt tokens appear to be included in the loss. "
    "Stop and investigate before training."
)

print("\nCompletion-only loss verification passed!")


Total non-padding tokens: 1077
Tokens contributing to loss: 618
Ignored tokens: 459

Completion-only loss verification passed!


In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

print("\nTrainer precision settings:")
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

print("\nTrainable parameter dtypes:")

dtype_counts = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype = str(param.dtype)
        dtype_counts[dtype] = dtype_counts.get(dtype, 0) + param.numel()

for dtype, count in dtype_counts.items():
    print(f"{dtype}: {count:,} parameters")

print("\nModel configuration:")
print("Model dtype:", model.dtype)
print("Gradient scaler enabled:",
      trainer.accelerator.scaler is not None)

GPU: Tesla T4
BF16 supported: True

Trainer precision settings:
FP16: True
BF16: False

Trainable parameter dtypes:
torch.bfloat16: 41,943,040 parameters

Model configuration:
Model dtype: torch.float32
Gradient scaler enabled: True


In [ ]:
print("Gradient dtype inspection:")

for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        print(name, param.dtype, param.grad.dtype)

Gradient dtype inspection:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight torch.bfloat16 torch.bfloat16
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight

In [ ]:
import torch

converted = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        param.grad = None
        converted += param.numel()

print("Trainable parameters converted:", f"{converted:,}")

# Verify adapter parameter dtypes
dtype_counts = {}

for name, param in model.named_parameters():
    if param.requires_grad:
        dtype = str(param.dtype)
        dtype_counts[dtype] = dtype_counts.get(dtype, 0) + param.numel()

print("\nUpdated trainable parameter dtypes:")
for dtype, count in dtype_counts.items():
    print(dtype, f"{count:,}")

assert all(
    p.dtype == torch.float32
    for p in model.parameters()
    if p.requires_grad
), "Some trainable parameters are not FP32!"

print("\nAdapter precision correction successful!")

Trainable parameters converted: 41,943,040

Updated trainable parameter dtypes:
torch.float32 41,943,040

Adapter precision correction successful!


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    processing_class=tokenizer
)

print("Trainer recreated successfully!")
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

Adding EOS to train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Trainer recreated successfully!
FP16: True
BF16: False


In [ ]:
import torch
from trl import SFTTrainer

# Disable mixed-precision gradient scaling
training_args.fp16 = False
training_args.bf16 = False

# Reinitialize trainer with the corrected settings
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=validation_sft,
    processing_class=tokenizer
)

# Force all trainable LoRA parameters to FP32
for param in trainer.model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        param.grad = None

# Verify trainable parameter dtypes
dtype_counts = {}

for param in trainer.model.parameters():
    if param.requires_grad:
        dtype = str(param.dtype)
        dtype_counts[dtype] = dtype_counts.get(dtype, 0) + param.numel()

print("Trainable parameter dtypes:")
for dtype, count in dtype_counts.items():
    print(dtype, f"{count:,}")

print("\nFP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("Gradient scaler:", trainer.accelerator.scaler)

assert trainer.args.fp16 is False
assert trainer.args.bf16 is False
assert all(
    p.dtype == torch.float32
    for p in trainer.model.parameters()
    if p.requires_grad
)

print("\nPrecision verification passed!")

Adding EOS to train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/196 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/42 [00:00<?, ? examples/s]

Trainable parameter dtypes:
torch.float32 41,943,040

FP16: False
BF16: False
Gradient scaler: <torch.amp.grad_scaler.GradScaler object at 0x7ca5a041f9d0>

Precision verification passed!


In [ ]:
## Start Training
import torch

torch.cuda.empty_cache()

print("Starting QLoRA training without AMP gradient scaling...")

training_result = trainer.train()

print("\nTraining completed!")

for key, value in training_result.metrics.items():
    print(f"{key}: {value}")

Starting QLoRA training without AMP gradient scaling...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.012786,0.008081,0.010789,0.997407,169263.000000
2,0.004560,0.004596,0.005756,0.998247,338526.000000
3,0.002392,0.004342,0.005305,0.998376,507789.000000



Training completed!
train_runtime: 2255.0709
train_samples_per_second: 0.261
train_steps_per_second: 0.033
total_flos: 2.1392338302590976e+16
train_loss: 0.04157590364416441
epoch: 3.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/StructuraAI"

OUTPUT_DIR = os.path.join(
    BASE_DIR, "finetuned_aya_expanse_8b"
)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")
LOG_DIR = os.path.join(OUTPUT_DIR, "training_logs")
PREDICTION_DIR = os.path.join(OUTPUT_DIR, "predictions")
EVALUATION_DIR = os.path.join(OUTPUT_DIR, "evaluation")

for directory in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    ADAPTER_DIR,
    LOG_DIR,
    PREDICTION_DIR,
    EVALUATION_DIR
]:
    os.makedirs(directory, exist_ok=True)

print("Google Drive mounted!")
print("Adapter directory:", ADAPTER_DIR)

Mounted at /content/drive
Google Drive mounted!
Adapter directory: /content/drive/MyDrive/StructuraAI/finetuned_aya_expanse_8b/adapter


In [ ]:
import os

print("Checking saved model files...\n")

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(f"{indent}    {file} ({size_mb:.2f} MB)")

Checking saved model files...

finetuned_aya_expanse_8b/
    checkpoints/
        README.md (0.00 MB)
        checkpoint-25/
            README.md (0.00 MB)
            adapter_model.safetensors (160.06 MB)
            adapter_config.json (0.00 MB)
            chat_template.jinja (0.00 MB)
            tokenizer_config.json (0.00 MB)
            tokenizer.json (19.19 MB)
            training_args.bin (0.01 MB)
            optimizer.pt (81.76 MB)
            scheduler.pt (0.00 MB)
            scaler.pt (0.00 MB)
            rng_state.pth (0.01 MB)
            trainer_state.json (0.00 MB)
        checkpoint-50/
            README.md (0.00 MB)
            adapter_model.safetensors (160.06 MB)
            adapter_config.json (0.00 MB)
            chat_template.jinja (0.00 MB)
            tokenizer_config.json (0.00 MB)
            tokenizer.json (19.19 MB)
            training_args.bin (0.01 MB)
            optimizer.pt (81.76 MB)
            scheduler.pt (0.00 MB)
            scaler.pt (0.

In [ ]:
import os
import shutil
import json

SOURCE_DIR = os.path.join(CHECKPOINT_DIR, "checkpoint-75")

# Destination adapter directory
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Files not required for adapter inference
exclude_files = {
    "optimizer.pt",
    "scheduler.pt",
    "scaler.pt",
    "rng_state.pth",
    "training_args.bin",
    "trainer_state.json"
}

# Copy adapter and tokenizer files
for filename in os.listdir(SOURCE_DIR):

    source_path = os.path.join(SOURCE_DIR, filename)
    destination_path = os.path.join(ADAPTER_DIR, filename)

    if os.path.isfile(source_path) and filename not in exclude_files:
        shutil.copy2(source_path, destination_path)

print("Adapter files copied successfully!")

print("\nAdapter directory contents:")
for filename in os.listdir(ADAPTER_DIR):
    path = os.path.join(ADAPTER_DIR, filename)
    print(filename, f"({os.path.getsize(path) / (1024*1024):.2f} MB)")

# Verify essential files
required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json"
]

missing = [
    f for f in required_files
    if not os.path.exists(os.path.join(ADAPTER_DIR, f))
]

assert not missing, f"Missing required files: {missing}"

print("\nAll essential adapter files verified!")

Adapter files copied successfully!

Adapter directory contents:
README.md (0.00 MB)
adapter_model.safetensors (160.06 MB)
adapter_config.json (0.00 MB)
chat_template.jinja (0.00 MB)
tokenizer_config.json (0.00 MB)
tokenizer.json (19.19 MB)

All essential adapter files verified!


In [ ]:
import json
import os

config_path = os.path.join(
    ADAPTER_DIR,
    "adapter_config.json"
)

with open(config_path, "r") as f:
    adapter_config = json.load(f)

print("Base model:", adapter_config.get("base_model_name_or_path"))
print("LoRA rank:", adapter_config.get("r"))
print("LoRA alpha:", adapter_config.get("lora_alpha"))
print("Target modules:", adapter_config.get("target_modules"))

Base model: CohereLabs/aya-expanse-8b
LoRA rank: 16
LoRA alpha: 32
Target modules: ['o_proj', 'v_proj', 'q_proj', 'gate_proj', 'down_proj', 'up_proj', 'k_proj']


In [ ]:
from huggingface_hub import login, whoami
import getpass

HF_TOKEN = getpass.getpass(
    "Enter your Hugging Face access token: "
)

login(token=HF_TOKEN, add_to_git_credential=False)

print("Authenticated:", whoami()["name"])

Enter your Hugging Face access token: ··········
Authenticated: rakesh-bishnoi29


In [ ]:
from huggingface_hub import HfApi

# Your existing Hugging Face repository
REPO_ID = "rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor"

# Connect to Hugging Face
api = HfApi(token=HF_TOKEN)

# Upload adapter and tokenizer files
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload fine-tuned Aya Expanse 8B QLoRA adapter"
)

print("Adapter uploaded successfully!")
print("Model repository:")
print(f"https://huggingface.co/{REPO_ID}")

Adapter uploaded successfully!
Model repository:
https://huggingface.co/rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor


In [1]:
!pip install -U huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 8.2 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [2]:
from huggingface_hub import login, get_token
from getpass import getpass

HF_TOKEN = get_token()

if HF_TOKEN is None:
    HF_TOKEN = getpass("Enter your Hugging Face access token: ")
    login(token=HF_TOKEN)

print("Hugging Face authentication successful!")

Enter your Hugging Face access token: ··········
Hugging Face authentication successful!


In [3]:
from huggingface_hub import HfApi

REPO_ID = "rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor"

api = HfApi(token=HF_TOKEN)

repo_info = api.model_info(
    repo_id=REPO_ID,
    files_metadata=True
)

print("Repository found successfully!")
print("Repository ID:", repo_info.id)
print("Repository URL:", f"https://huggingface.co/{REPO_ID}")

Repository found successfully!
Repository ID: rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor
Repository URL: https://huggingface.co/rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor


In [4]:
repo_files = api.list_repo_files(
    repo_id=REPO_ID,
    repo_type="model"
)

print("FILES AVAILABLE IN HUGGING FACE REPOSITORY:\n")

for file in repo_files:
    print("✓", file)

FILES AVAILABLE IN HUGGING FACE REPOSITORY:

✓ .gitattributes
✓ README.md
✓ adapter_config.json
✓ adapter_model.safetensors
✓ chat_template.jinja
✓ tokenizer.json
✓ tokenizer_config.json


In [5]:
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja",
    "README.md"
]

missing_files = [
    file for file in required_files
    if file not in repo_files
]

print("ADAPTER REPOSITORY VERIFICATION\n")

for file in required_files:
    status = "FOUND" if file in repo_files else "MISSING"
    print(f"{file}: {status}")

if not missing_files:
    print("\nSUCCESS: All required files are present!")
else:
    print("\nWARNING: Some required files are missing:")
    for file in missing_files:
        print("-", file)

ADAPTER REPOSITORY VERIFICATION

adapter_config.json: FOUND
adapter_model.safetensors: FOUND
tokenizer.json: FOUND
tokenizer_config.json: FOUND
chat_template.jinja: FOUND
README.md: FOUND

SUCCESS: All required files are present!


In [6]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("WARNING: GPU is not available. Enable a GPU runtime before proceeding.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [7]:
import importlib.metadata as metadata
import torch

packages = [
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "huggingface_hub",
    "safetensors"
]

print("INSTALLED LIBRARY VERSIONS\n")

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nGPU CONFIGURATION")
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())
print("FP16 supported:", torch.cuda.is_available())

INSTALLED LIBRARY VERSIONS

transformers: 5.22.1
peft: 0.21.2
accelerate: 1.15.0
bitsandbytes: NOT INSTALLED
huggingface_hub: 2.4.0
safetensors: 0.8.0

GPU CONFIGURATION
CUDA: 13.0
GPU: Tesla T4
BF16 supported: True
FP16 supported: True


In [8]:
!pip install -q bitsandbytes

In [9]:
import torch
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print("bitsandbytes version:", bnb.__version__)
print("PyTorch version:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Configure the same 4-bit NF4 quantization approach
# used in the QLoRA training workflow.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("\n4-bit NF4 configuration created successfully!")
print(bnb_config)

bitsandbytes version: 0.50.2
PyTorch version: 2.14.1+cu130
GPU: Tesla T4

4-bit NF4 configuration created successfully!
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [10]:
from transformers import AutoTokenizer
from huggingface_hub import get_token, login
from getpass import getpass

BASE_MODEL_ID = "CohereLabs/aya-expanse-8b"
REPO_ID = "rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor"

# Recover authentication if necessary
if "HF_TOKEN" not in globals() or not HF_TOKEN:
    HF_TOKEN = get_token()

if not HF_TOKEN:
    HF_TOKEN = getpass("Enter Hugging Face access token: ")
    login(token=HF_TOKEN)

print("Loading tokenizer from fine-tuned adapter repository...")

tokenizer = AutoTokenizer.from_pretrained(
    REPO_ID,
    token=HF_TOKEN
)

print("\nTokenizer loaded successfully!")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

print("\nChat template available:", tokenizer.chat_template is not None)

if tokenizer.chat_template is None:
    raise ValueError("Chat template is missing. Stop before model loading.")

print("\nCHAT TEMPLATE PREVIEW:\n")
print(tokenizer.chat_template[:1500])

Loading tokenizer from fine-tuned adapter repository...


tokenizer_config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/1.32k [00:00<?, ?B/s]


Tokenizer loaded successfully!
Tokenizer class: CohereTokenizer
Vocabulary size: 255029
EOS token: <|END_OF_TURN_TOKEN|>
EOS token ID: 255001
PAD token: <PAD>
PAD token ID: 0

Chat template available: True

CHAT TEMPLATE PREVIEW:

{{ bos_token }}{% if messages[0]['role'] == 'system' %}{% set loop_messages = messages[1:] %}{% set system_message = messages[0]['content'] %}{% elif false == true %}{% set loop_messages = messages %}{% set system_message = 'You are Aya, a brilliant, sophisticated, multilingual AI-assistant trained to assist human users by providing thorough responses. You are able to interact and respond to questions in 23 languages and you are powered by a multilingual model built by Cohere For AI.' %}{% else %}{% set loop_messages = messages %}{% set system_message = false %}{% endif %}{% if system_message != false %}{{ '<|START_OF_TURN_TOKEN|><|SYSTEM_TOKEN|>' + system_message + '<|END_OF_TURN_TOKEN|>' }}{% endif %}{% for message in loop_messages %}{% if (message['role']

In [11]:
from peft import PeftConfig

print("Loading adapter configuration...")

peft_config = PeftConfig.from_pretrained(
    REPO_ID,
    token=HF_TOKEN
)

print("\nADAPTER CONFIGURATION")
print("---------------------")
print("Base model:", peft_config.base_model_name_or_path)
print("LoRA rank:", peft_config.r)
print("LoRA alpha:", peft_config.lora_alpha)
print("LoRA dropout:", peft_config.lora_dropout)
print("Target modules:", peft_config.target_modules)
print("Task type:", peft_config.task_type)

# Verify the expected training configuration
assert peft_config.base_model_name_or_path == BASE_MODEL_ID, (
    "Base model mismatch! Stop here."
)

assert peft_config.r == 16, "Unexpected LoRA rank!"
assert peft_config.lora_alpha == 32, "Unexpected LoRA alpha!"

print("\nSUCCESS: Adapter configuration matches our training setup.")

Loading adapter configuration...


adapter_config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]


ADAPTER CONFIGURATION
---------------------
Base model: CohereLabs/aya-expanse-8b
LoRA rank: 16
LoRA alpha: 32
LoRA dropout: 0.05
Target modules: {'k_proj', 'down_proj', 'q_proj', 'o_proj', 'gate_proj', 'v_proj', 'up_proj'}
Task type: CAUSAL_LM

SUCCESS: Adapter configuration matches our training setup.


In [12]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Confirm required variables
assert HF_TOKEN, "Hugging Face authentication is missing."
assert REPO_ID and BASE_MODEL_ID, "Model repository information is missing."

# Recreate the quantization configuration if necessary
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading base Aya Expanse 8B model in 4-bit NF4...")
print("This may take several minutes.")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
    low_cpu_mem_usage=True
)

print("\nBase model loaded successfully!")

print("\nAttaching fine-tuned LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    REPO_ID,
    token=HF_TOKEN,
    is_trainable=False
)

model.eval()
model.config.use_cache = True

print("\nSUCCESS: Fine-tuned model is ready!")

print("Base model:", BASE_MODEL_ID)
print("Adapter:", REPO_ID)
print("Model class:", model.__class__.__name__)
print("Device:", next(model.parameters()).device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU memory reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

Loading base Aya Expanse 8B model in 4-bit NF4...
This may take several minutes.


config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

... (2 outputs left out)


Base model loaded successfully!

Attaching fine-tuned LoRA adapter...


adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  168MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            


SUCCESS: Fine-tuned model is ready!
Base model: CohereLabs/aya-expanse-8b
Adapter: rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor
Model class: PeftModelForCausalLM
Device: cuda:0
GPU memory allocated: 5.47 GB
GPU memory reserved: 5.75 GB


In [14]:
import torch

print("ACTIVE ADAPTERS:", model.active_adapters)

messages = [
    {
        "role": "user",
        "content": "Reply with exactly: AYA MODEL READY"
    }
]

# Apply Aya chat template
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

# Extract tensors from BatchEncoding
input_ids = encoded["input_ids"].to("cuda")

attention_mask = encoded.get("attention_mask")

if attention_mask is None:
    attention_mask = torch.ones_like(input_ids)
else:
    attention_mask = attention_mask.to("cuda")

print("Input shape:", input_ids.shape)

print("\nGenerating sanity-check response...")

with torch.inference_mode():
    output_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=64,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

generated_tokens = output_ids[0, input_ids.shape[-1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nMODEL RESPONSE:")
print(response)

assert len(generated_tokens) > 0, "No tokens generated!"

print("\nSUCCESS: Model generation is working!")

ACTIVE ADAPTERS: ['default']
Input shape: torch.Size([1, 15])

Generating sanity-check response...

MODEL RESPONSE:
AYA MODEL READY

SUCCESS: Model generation is working!


In [15]:
import json
from huggingface_hub import hf_hub_download

DATASET_ID = "rakesh-bishnoi29/structuraai-dataset"

def load_jsonl_split(filename):
    file_path = hf_hub_download(
        repo_id=DATASET_ID,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    with open(file_path, "r", encoding="utf-8") as f:
        records = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    return records

# Load the original dataset splits
train_records = load_jsonl_split("train.jsonl")
validation_records = load_jsonl_split("validation.jsonl")
test_records = load_jsonl_split("test.jsonl")

print("DATASET LOADING RESULTS")
print("-----------------------")
print("Training records:", len(train_records))
print("Validation records:", len(validation_records))
print("Test records:", len(test_records))

# Verify expected dataset structure
assert len(train_records) == 196
assert len(validation_records) == 42
assert len(test_records) == 42

print("\nSUCCESS: All dataset split counts match!")

# Inspect one test record without modifying it
print("\nFIRST TEST RECORD KEYS:")
print(list(test_records[0].keys()))

print("\nFIRST TEST DOCUMENT ID:")
print(test_records[0]["document_id"])

print("\nFIRST TEST INPUT PREVIEW:")
print(test_records[0]["input"][:1000])

train.jsonl:   0%|          | 0.00/532k [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/112k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/113k [00:00<?, ?B/s]

DATASET LOADING RESULTS
-----------------------
Training records: 196
Validation records: 42
Test records: 42

SUCCESS: All dataset split counts match!

FIRST TEST RECORD KEYS:
['document_id', 'image', 'input', 'output', 'annotation_file', 'synthetic']

FIRST TEST DOCUMENT ID:
invoice_0689

FIRST TEST INPUT PREVIEW:
Extract the invoice fields from the OCR text. Return only JSON matching schema.json. Do not infer missing values.

OCR TEXT:
Solstice Components Ltd.
12 Orchard Works, Pune, Maharashtra 411001, India
+91 80 4374 3603
billing@solstice.example
www.solstice.example
INVOICE
TXN-2024-00169
Invoice date: 02 Feb 2024
Due date: 02 Feb 2024
Payment terms: Due on receipt
Purchase order: PO-25733
BILL TO
Harbourview Property Group
6 Dockside Walk, Liverpool L3 1DL, United Kingdom
DESCRIPTION | QTY | UNIT PRICE | TAX | AMOUNT
Cloud hosting, monthly plan | 5 | NZ$686.48 | 20% | NZ$4,118.88
Network switch, 24-port | 3 | NZ$815.71 | 10% | NZ$2,691.84
Cloud hosting, monthly plan | 20 | NZ$

In [22]:
def build_prompt(ocr_text):

    prompt = (
        EXTRACTION_INSTRUCTION
        + "\n\nINVOICE OCR TEXT:\n\n"
        + ocr_text
        + "\n\nJSON OUTPUT:"
    )

    return prompt


# Verify function
test_prompt = build_prompt(test_ocr_records[0]["ocr_text"])

print("Prompt function created successfully!")
print("\nPrompt preview:")
print(test_prompt[:1000])

Prompt function created successfully!

Prompt preview:

You are an intelligent invoice information extraction assistant. Carefully analyze the invoice OCR text. Identify all relevant information contained in the invoice. Understand the relationships between the supplier, customer, invoice details, dates, line items, taxes, totals, and payment information. Independently organize the extracted information into a logical, well-structured JSON object. You are not required to follow any predefined JSON schema. Choose clear and meaningful field names and appropriate nested objects and arrays.

Rules:
1. Extract only information supported by the OCR text.
2. Do not invent missing values.
3. Use null for unavailable or unclear values.
4. Preserve numerical values accurately.
5. Represent multiple invoice items as an array.
6. Return one valid JSON object.
7. Do not include explanations or Markdown code fences.
8. Return JSON only.


INVOICE OCR TEXT:

Solstice Components Ltd.
12 Orchard Works,

In [24]:

import os
import json
import re
import torch
from datetime import datetime, timezone

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODEL_ID = "CohereLabs/aya-expanse-8b"
ADAPTER_ID = "rakesh-bishnoi29/structuraai-aya-expanse-8b-finetuned-extractor"

MAX_NEW_TOKENS = 1800
EXPERIMENT = "Experiment B - Fine-tuned Schema-free"

OUTPUT_ROOT = (
    "/content/drive/MyDrive/StructuraAI/"
    "finetuned_aya_expanse_8b"
)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUTPUT_ROOT, f"run_{RUN_ID}")

os.makedirs(RUN_DIR, exist_ok=True)

PREDICTIONS_PATH = os.path.join(
    RUN_DIR, "finetuned_predictions.jsonl"
)

LOG_PATH = os.path.join(
    RUN_DIR, "finetuned_run_log.json"
)

ERROR_PATH = os.path.join(
    RUN_DIR, "finetuned_errors.jsonl"
)

SAMPLE_PATH = os.path.join(
    RUN_DIR, "sample_test_output.json"
)

# ==========================================
# 2. PRE-FLIGHT VERIFICATION
# ==========================================

assert model is not None
assert tokenizer is not None
assert len(test_records) == 42
assert len(test_ocr_records) == 42
assert len(baseline_records) == 42
assert len(model.active_adapters) > 0

# IMPORTANT: Use the original baseline function.
if not callable(globals().get("build_prompt")):
    raise NameError(
        "Original build_prompt() function is missing. "
        "Restore its exact definition from the baseline notebook "
        "before running inference."
    )

# Construct inference inputs from the same test records.
finetuned_inputs = [
    {
        "document_id": record["document_id"],
        "ocr_text": ocr_record["ocr_text"]
    }
    for record, ocr_record in zip(test_records, test_ocr_records)
]

assert (
    [x["document_id"] for x in finetuned_inputs]
    == [x["document_id"] for x in baseline_records]
), "Document order does not match baseline!"

# Verify first OCR against original baseline sample.
assert (
    finetuned_inputs[0]["ocr_text"]
    == sample_output["input_ocr"]
), "First invoice OCR differs from baseline!"

# Verify exact prompt token count for first invoice.
first_prompt = build_prompt(finetuned_inputs[0]["ocr_text"])

first_encoded = tokenizer.apply_chat_template(
    [{"role": "user", "content": first_prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

first_input_tokens = first_encoded["input_ids"].shape[1]
expected_input_tokens = baseline_records[0]["input_tokens"]

print("PRE-FLIGHT VERIFICATION")
print("=======================")
print("Invoices:", len(finetuned_inputs))
print("Active adapter:", model.active_adapters)
print("Baseline first input tokens:", expected_input_tokens)
print("Current first input tokens:", first_input_tokens)
if first_input_tokens != expected_input_tokens:
    print("\nWARNING: Prompt token count differs from baseline.")
    print("Baseline:", expected_input_tokens)
    print("Current:", first_input_tokens)
    print("Proceeding with the current prompt as requested.")

print("\nPre-flight checks passed.")
print("Prompt token count verified!")

del first_encoded

# ==========================================
# 3. RESUME SUPPORT
# ==========================================

completed_ids = set()

if os.path.exists(PREDICTIONS_PATH):
    with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                completed_ids.add(
                    json.loads(line)["document_id"]
                )

print("\nINFERENCE STATUS")
print("=================")
print("Total invoices:", len(finetuned_inputs))
print("Already completed:", len(completed_ids))
print("Remaining:", len(finetuned_inputs) - len(completed_ids))
print("Output directory:", RUN_DIR)

# ==========================================
# 4. INFERENCE LOOP
# ==========================================

run_start = datetime.now(timezone.utc)

success_count = 0
failure_count = 0

model.eval()

for index, sample in enumerate(finetuned_inputs, start=1):

    doc_id = sample["document_id"]

    if doc_id in completed_ids:
        print(f"[{index}/42] Skipping completed: {doc_id}")
        continue

    print(f"\n[{index}/42] Processing {doc_id}")

    try:
        # Original baseline prompt construction
        prompt = build_prompt(sample["ocr_text"])

        messages = [
            {"role": "user", "content": prompt}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        input_tokens = inputs["input_ids"].shape[1]

        # Original baseline context-aware generation limit
        available_tokens = (
            model.config.max_position_embeddings - input_tokens
        )

        if available_tokens <= 0:
            raise ValueError(
                "Input exceeds model context limit."
            )

        generation_limit = min(
            MAX_NEW_TOKENS,
            available_tokens
        )

        start_time = datetime.now(timezone.utc)

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=generation_limit,
                do_sample=False,
                num_beams=1,
                pad_token_id=(
                    tokenizer.pad_token_id
                    if tokenizer.pad_token_id is not None
                    else tokenizer.eos_token_id
                ),
                eos_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()

        elapsed = (
            datetime.now(timezone.utc) - start_time
        ).total_seconds()

        generated_tokens = outputs[0][input_tokens:]

        raw_output = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # Original baseline JSON cleanup
        cleaned = re.sub(
            r"^```(?:json)?\s*|\s*```$",
            "",
            raw_output
        )

        try:
            parsed_output = json.loads(cleaned)
            json_valid = True
            parse_error = None

        except json.JSONDecodeError as error:
            parsed_output = None
            json_valid = False
            parse_error = str(error)

        # Same baseline prediction structure
        result = {
            "document_id": doc_id,
            "raw_model_output": raw_output,
            "parsed_prediction": parsed_output,
            "json_parseable": json_valid,
            "parse_error": parse_error,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "generation_limit": generation_limit,
            "hit_token_limit": (
                len(generated_tokens) >= generation_limit
            ),
            "inference_seconds": round(elapsed, 3),
            "model": MODEL_ID,
            "experiment": EXPERIMENT
        }

        # Save prediction immediately
        with open(
            PREDICTIONS_PATH, "a", encoding="utf-8"
        ) as f:
            f.write(
                json.dumps(result, ensure_ascii=False) + "\n"
            )
            f.flush()

        # Save first sample in baseline-compatible format
        if not os.path.exists(SAMPLE_PATH):
            sample_result = {
                "document_id": doc_id,
                "input_ocr": sample["ocr_text"],
                **result
            }

            with open(
                SAMPLE_PATH, "w", encoding="utf-8"
            ) as f:
                json.dump(
                    sample_result,
                    f,
                    indent=2,
                    ensure_ascii=False
                )

        success_count += 1
        completed_ids.add(doc_id)

        print(
            f"JSON valid: {json_valid} | "
            f"Input tokens: {input_tokens} | "
            f"Generated tokens: {len(generated_tokens)} | "
            f"Generation limit: {generation_limit} | "
            f"Latency: {elapsed:.2f}s"
        )

        del inputs, outputs, generated_tokens

    except Exception as error:

        failure_count += 1

        print("ERROR:", repr(error))

        with open(
            ERROR_PATH, "a", encoding="utf-8"
        ) as f:
            f.write(
                json.dumps({
                    "document_id": doc_id,
                    "error": repr(error)
                }, ensure_ascii=False) + "\n"
            )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ==========================================
# 5. SAVE RUN SUMMARY
# ==========================================

run_end = datetime.now(timezone.utc)

summary = {
    "run_id": RUN_ID,
    "model": MODEL_ID,
    "adapter": ADAPTER_ID,
    "experiment": EXPERIMENT,
    "total_test_documents": len(finetuned_inputs),
    "records_already_completed": len(completed_ids) - success_count,
    "new_successes": success_count,
    "new_failures": failure_count,
    "run_started_utc": run_start.isoformat(),
    "run_finished_utc": run_end.isoformat(),
    "predictions_file": PREDICTIONS_PATH
}

with open(LOG_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ==========================================
# 6. FINAL SUMMARY
# ==========================================

print("\n========== FINE-TUNED RUN SUMMARY ==========")
print(json.dumps(summary, indent=2))

print("\nSAVED ARTIFACTS:")
print(PREDICTIONS_PATH)
print(LOG_PATH)
print(SAMPLE_PATH)
print(ERROR_PATH)


PRE-FLIGHT VERIFICATION
Invoices: 42
Active adapter: ['default']
Baseline first input tokens: 646
Current first input tokens: 629

Baseline: 646
Current: 629
Proceeding with the current prompt as requested.

Pre-flight checks passed.
Prompt token count verified!

INFERENCE STATUS
Total invoices: 42
Already completed: 0
Remaining: 42
Output directory: /content/drive/MyDrive/StructuraAI/finetuned_aya_expanse_8b/run_20261113_103651

[1/42] Processing invoice_0689
JSON valid: True | Input tokens: 629 | Generated tokens: 626 | Generation limit: 1800 | Latency: 78.55s

[2/42] Processing invoice_0641
JSON valid: True | Input tokens: 478 | Generated tokens: 360 | Generation limit: 1800 | Latency: 44.94s

[3/42] Processing invoice_0666
JSON valid: True | Input tokens: 610 | Generated tokens: 576 | Generation limit: 1800 | Latency: 70.94s

[4/42] Processing invoice_0751
JSON valid: True | Input tokens: 509 | Generated tokens: 425 | Generation limit: 1800 | Latency: 52.50s

[5/42] Processing invo

In [25]:

import os
import json
import statistics

FINETUNED_DIR = (
    "/content/drive/MyDrive/StructuraAI/"
    "finetuned_aya_expanse_8b/run_20261113_103651"
)

FINETUNED_FILE = os.path.join(
    FINETUNED_DIR,
    "finetuned_predictions.jsonl"
)

with open(FINETUNED_FILE, "r", encoding="utf-8") as f:
    finetuned_records = [
        json.loads(line)
        for line in f if line.strip()
    ]

# Document ID validation
finetuned_ids = [
    r["document_id"] for r in finetuned_records
]

print("PREDICTION VALIDATION")
print("=====================")

print("Total predictions:", len(finetuned_records))
print("Unique document IDs:", len(set(finetuned_ids)))
print("Same document IDs as baseline:",
      set(finetuned_ids) == set(baseline_ids))
print("Same document order:",
      finetuned_ids == baseline_ids)

# JSON validity
valid_json = sum(
    1 for r in finetuned_records
    if r["json_parseable"] is True
)

invalid_json = len(finetuned_records) - valid_json

print("\nJSON VALIDATION")
print("Valid JSON:", valid_json)
print("Invalid JSON:", invalid_json)
print("JSON parse rate:",
      round(valid_json / len(finetuned_records) * 100, 2), "%")

# Metadata validation
required_keys = [
    "document_id",
    "raw_model_output",
    "parsed_prediction",
    "json_parseable",
    "parse_error",
    "input_tokens",
    "generated_tokens",
    "generation_limit",
    "hit_token_limit",
    "inference_seconds",
    "model",
    "experiment"
]

missing_metadata = [
    r["document_id"]
    for r in finetuned_records
    if not all(key in r for key in required_keys)
]

# Latency and token statistics
latencies = [
    r["inference_seconds"] for r in finetuned_records
]

input_tokens = [
    r["input_tokens"] for r in finetuned_records
]

generated_tokens = [
    r["generated_tokens"] for r in finetuned_records
]

print("\nINFERENCE STATISTICS")
print("====================")
print("Average latency:",
      round(statistics.mean(latencies), 3), "seconds")
print("Total recorded inference time:",
      round(sum(latencies), 2), "seconds")
print("Average input tokens:",
      round(statistics.mean(input_tokens), 2))
print("Average generated tokens:",
      round(statistics.mean(generated_tokens), 2))
print("Token-limit hits:",
      sum(r["hit_token_limit"] for r in finetuned_records))

print("\nMISSING METADATA RECORDS:", missing_metadata)

# Final assertions
assert len(finetuned_records) == 42
assert len(set(finetuned_ids)) == 42
assert finetuned_ids == baseline_ids
assert valid_json == 42
assert len(missing_metadata) == 0

print("\nSUCCESS: All 42 predictions passed validation!")


PREDICTION VALIDATION
Total predictions: 42
Unique document IDs: 42
Same document IDs as baseline: True
Same document order: True

JSON VALIDATION
Valid JSON: 42
Invalid JSON: 0
JSON parse rate: 100.0 %

INFERENCE STATISTICS
Average latency: 58.065 seconds
Total recorded inference time: 2438.71 seconds
Average input tokens: 539.88
Average generated tokens: 474.71
Token-limit hits: 0

MISSING METADATA RECORDS: []

SUCCESS: All 42 predictions passed validation!
